# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 01: Distributed Data Parallelism (DDP) & Ring-AllReduce from Scratch

### 1. Conceptual Intuition & Real-World Use Case
When training modern deep learning models, training on a single GPU is limited by compute and memory bandwidth.

**Distributed Data Parallelism (DDP)** runs a replica of the full model on each GPU (rank):
1. Each GPU receives a distinct partition of the training batch.
2. Each GPU computes forward pass and backward gradients independently.
3. Crucially, before updating weights, all GPUs must **average their gradients** so models stay in sync.

The naive approach (sending all gradients to GPU 0 as a master server) creates a catastrophic network bottleneck. Modern distributed systems (PyTorch DDP, NCCL) solve this using the **Ring-AllReduce** algorithm!

---

### 2. Mathematical Foundations
Let $N$ be the number of GPUs (ranks), and $S$ be the total gradient size in bytes.

1. **Master-Worker Parameter Server Bottleneck:**
   GPU 0 receives $(N-1) S$ bytes and broadcasts $(N-1) S$ bytes. Total network volume scales linearly with $N$, saturating GPU 0.

2. **Ring-AllReduce Communication Complexity:**
   GPUs are logically arranged in a unidirectional ring: $\text{GPU}_0 \to \text{GPU}_1 \to \dots \to \text{GPU}_{N-1} \to \text{GPU}_0$.
   The gradient tensor is partitioned into $N$ equal chunks of size $S / N$.
   
   - **Phase 1: Scatter-Reduce ($N-1$ steps):**
     In each step, GPU $i$ sends chunk $k$ to GPU $i+1$ and receives chunk $k-1$ from GPU $i-1$, summing it with its local chunk.
     After $N-1$ steps, each GPU holds the global sum of exactly one chunk!
   
   - **Phase 2: Allgather ($N-1$ steps):**
     In each step, the fully summed chunks are circulated around the ring until all GPUs hold all summed chunks.

3. **Total Transferred Data per GPU:**
   $$\text{Bytes Transferred} = 2 \times \left( \frac{N - 1}{N} \right) \times S$$
   As $N$ grows large, communication per GPU approaches $2S$, which is **independent of the number of GPUs $N$**! This enables linear scaling to thousands of GPUs.

In [ ]:
import numpy as np

# Step 1: Simulate 4 GPUs (ranks) participating in Ring-AllReduce
num_gpus = 4
tensor_size = 8  # 8 gradient parameters

# Each GPU has its own locally computed gradient tensor
# Partitioned into 4 chunks (each chunk has length 2)
np.random.seed(10)
gradients = {
    rank: np.random.randint(1, 10, size=tensor_size).astype(float)
    for rank in range(num_gpus)
}

print("Initial Gradients across 4 GPUs:")
for rank in range(num_gpus):
    print(f"  GPU {rank}: {gradients[rank]}")

# Expected exact global sum across all ranks
expected_sum = np.sum([gradients[r] for r in range(num_gpus)], axis=0)
print(f"\nExpected Global Sum Target: {expected_sum}")

# Step 2: Split each GPU's tensor into num_gpus chunks
chunk_size = tensor_size // num_gpus
gpu_chunks = {
    rank: [gradients[rank][c * chunk_size : (c + 1) * chunk_size].copy() for c in range(num_gpus)]
    for rank in range(num_gpus)
}

# Step 3: Phase 1 - Scatter-Reduce (num_gpus - 1 steps)
print("\n--- Phase 1: Scatter-Reduce ---")
for step in range(num_gpus - 1):
    # Buffer to hold incoming chunks
    incoming = {}
    for rank in range(num_gpus):
        send_to = (rank + 1) % num_gpus
        chunk_idx = (rank - step) % num_gpus
        incoming[send_to] = (chunk_idx, gpu_chunks[rank][chunk_idx].copy())
        
    for rank in range(num_gpus):
        chunk_idx, data = incoming[rank]
        gpu_chunks[rank][chunk_idx] += data
    print(f"Step {step + 1} completed.")

# Step 4: Phase 2 - Allgather (num_gpus - 1 steps)
print("\n--- Phase 2: Allgather ---")
for step in range(num_gpus - 1):
    incoming = {}
    for rank in range(num_gpus):
        send_to = (rank + 1) % num_gpus
        chunk_idx = (rank - step + 1) % num_gpus
        incoming[send_to] = (chunk_idx, gpu_chunks[rank][chunk_idx].copy())
        
    for rank in range(num_gpus):
        chunk_idx, data = incoming[rank]
        gpu_chunks[rank][chunk_idx] = data
    print(f"Step {step + 1} completed.")

# Step 5: Reassemble chunks on each GPU
final_allreduce_result = {}
for rank in range(num_gpus):
    final_allreduce_result[rank] = np.concatenate(gpu_chunks[rank])

print("\nFinal AllReduce Results on Each GPU:")
for rank in range(num_gpus):
    is_match = np.allclose(final_allreduce_result[rank], expected_sum)
    print(f"  GPU {rank} Output: {final_allreduce_result[rank]} | Matches Target: {is_match}")

### 3. Step-by-Step Code Walkthrough
1. **Ring Topology:** Ranks communicate strictly with immediate neighbors: $\text{rank} \to (\text{rank} + 1) \% N$.
2. **Scatter-Reduce:** Each step transmits a different chunk, accumulating gradients incrementally until every rank owns one fully reduced chunk.
3. **Allgather:** The fully reduced chunks are propagated in a ring until all ranks have the complete averaged gradient vector.
4. **Bandwidth Efficiency:** No single GPU is overwhelmed. All physical PCIe / NVLink communication channels operate at 100% saturation simultaneously.

---

### 4. Industry Applications & Trade-offs
- **PyTorch `DistributedDataParallel` (DDP):** Overlaps backward computation with communication using gradient buckets.
- **Scaling Limit:** DDP requires the **entire model and optimizer state to fit into every single GPU's VRAM**. When models exceed 10B+ parameters, Model/Tensor Parallelism is required.

---

### 5. Top Interview Questions & Answers
**Q1: Why is PyTorch DDP vastly faster than PyTorch DataParallel (DP)?**
> **Answer:** DataParallel uses single-process multi-threading subject to Python's Global Interpreter Lock (GIL) and sends all gradients through GPU 0. DDP uses multi-process execution (one process per GPU, no GIL) and Ring-AllReduce via NCCL, overlapping communication with backward gradient passes.

**Q2: What is "gradient bucketing" in PyTorch DDP?**
> **Answer:** Launching AllReduce for every individual layer parameter creates massive CUDA kernel launch overhead. PyTorch DDP groups multiple layer gradients into contiguous 25MB buckets and fires an AllReduce call the moment a bucket fills up, overlapping network communication with backward pass computation of earlier layers.